# Course 2: Sentiment Analysis using Logistic Regression
In this tutorial you'll learn how to build a sentiment analysis model using logistic regression: 

* Learn how to extract features for logistic regression given some text
* Implement logistic regression using only mathematical formula without using any python library
* Apply logistic regression on a natural language processing task
* Validate your inference performance

## Table of Contents

- [Import Libraries and Data](#0)
- [1 - Extracting the Features](#1)
- [2 - Logistic Regression](#1)
    - [2.1 - Sigmoid function](#1-1)
    - [2.2 - Cost function and Gradient](#1-2)
- [3 - Training Your Model](#3)
- [4 - Test your Logistic Regression](#4)
    - [4.1 - Check the Performance using the Test Set](#4-1)
- [5 - Predict with your own text](#5)

<a name='0'></a>
## Import Libraries and Data

In [1]:
#!pip install nltk

In [2]:
#import nltk
import nltk
from os import getcwd
import numpy as np
import pandas as pd
nltk.download()
nltk.download('stopwords')

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gabriel/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Imported Libraries

Download the data needed current tutorials. Feel free to doanload all data sets from nltk corpora. For additional information feel free to check for documentation https://www.nltk.org/book/.

* to use entire data sets you will need to download it using:
```Python
nltk.download()
```

* stopwords: to run this notebook on your local computer, you will need to download it using:
```python
nltk.download('stopwords')
```

In [3]:
#set the file path
filePath = f"{getcwd()}"
nltk.data.path.append(filePath)

In [4]:
# choose one dataset from nltk corpora
from nltk.corpus import pros_cons

### Prepare the Data
* The pros_cons contains subsets of more than 20K reviews for each class.  

In [5]:
# read the pros and cons reviews
reviews_cons = pros_cons.raw('IntegratedCons.txt')
reviews_pros = pros_cons.raw('IntegratedPros.txt')
reviews_pros

'        <Pros>Easy to use, economical!</Pros>\n        <Pros>Digital is where it\'s at...down with developing film!</Pros>\n        <Pros>Good image quality, 3x optical zoom, macro mode, inexpensive</Pros>\n        <Pros>Awesome features/easy to use/fun/versatile/low price/Cust SVS 2nd 2 none!</Pros>\n        <Pros>intuitive, user friendly</Pros>\n        <Pros>Simple, Flexable, Reliable</Pros>\n        <Pros>battery life, download speed</Pros>\n        <Pros>Agfa quality in a small box, great for table top and tests</Pros>\n        <Pros>Image quality, price, video link</Pros>\n        <Pros>No Extras to Buy, Very Flexible, High Quality Images, Good Software</Pros>\n        <Pros>simplicity</Pros>\n        <Pros>Comes with software, good price, excellent pictures.</Pros>\n        <Pros>Clear pics</Pros>\n        <Pros>Price, Over all Performance, Zoom Feature</Pros>\n        <Pros>clear, srisp picture, durable camera, good desing, nice features</Pros>\n        <Pros>Inexpensive compa

In [6]:
#remove begin,end tags <Pros>, <Cons>
reviews_cons = reviews_cons.replace('<Cons>','').replace('</Cons>','').strip()
reviews_pros = reviews_pros.replace('<Pros>','').replace('</Pros>','').strip()
reviews_cons_list = reviews_cons.split('\n')
reviews_pros_list = reviews_pros.split('\n')

* Train set 80% and test set 20%.


In [7]:
train_pos = reviews_pros_list[:16000]
test_pos = reviews_pros_list[16000:20000]
train_neg = reviews_cons_list[:16000]
test_neg = reviews_cons_list[16000:20000]
train_x = train_pos + train_neg 
test_x = test_pos + test_neg

* Create the numpy array of positive labels and negative labels.

In [8]:
# combine positive and negative labels
train_x = train_pos + train_neg 
test_x = test_pos + test_neg
train_y = np.append(np.ones((len(train_pos), 1)), np.zeros((len(train_neg), 1)), axis=0)
test_y = np.append(np.ones((len(test_pos), 1)), np.zeros((len(test_neg), 1)), axis=0)

In [9]:
# Print the shape train and test sets
print("train_y.shape = " + str(train_y.shape))
print("test_y.shape = " + str(test_y.shape))

train_y.shape = (32000, 1)
test_y.shape = (8000, 1)


* Create the function for processing the string/text:
    - tokenization.
    - remove stop words.
    - apply stemming.  

In [10]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

def process_text(str):
    """Process text function.
    Inputs:
        text: a string containing a text
    Output:
        tokens_clean: a list of words containing the processed string

    """
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # remove hyperlinks    
    str = re.sub(r'https?://[^\s\n\r]+', '', str)
    # remove hashtags
    # only removing the hash # sign from the word
    str = re.sub(r'#', '', str)
    # tokenize tweets
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)
    str_tokens = tokenizer.tokenize(str)

    tokens_clean = []
    for word in str_tokens:
        if (word not in stopwords_english and  # remove stopwords
                word not in string.punctuation):  # remove punctuation
            stem_word = stemmer.stem(word)  # stemming word
            tokens_clean.append(stem_word)

    return tokens_clean

* Create the frequency dictionary function.  

In [11]:
def build_freqs(strs, ys):
    """
    Inputs: strs - a list of strings
            ys - an mx1 array with the label (0 or 1) for each word
    Outputs: freqs - a dictionary of pairs of each word and label
    """
    yslist = np.squeeze(ys).tolist()
    # initialize an empty dictionary
    freqs={}
    for y, str in zip(yslist, strs):
        for word in process_text(str):
            pair = (word, y)
            if pair in freqs:
                freqs[pair] += 1
            else:
                freqs[pair] = 1
    return freqs

In [15]:
first_key = next(iter(freqs))
first_value = freqs[first_key]

print(first_key, first_value)

('easi', 1.0) 3806


In [16]:
token = "easi"

pos = freqs.get((token,1),0)
neg = freqs.get((token,0),0)

print(pos, neg)

3806 98


In [12]:
# create frequency dictionary
freqs = build_freqs(train_x, train_y)
# check the output
print("type(freqs) = " + str(type(freqs)))
print("len(freqs) = " + str(len(freqs.keys())))

type(freqs) = <class 'dict'>
len(freqs) = 8884


### Process Text
The given function 'process_text' tokenizes the review into individual words, removes stop words and applies stemming.

In [17]:
# test the function below
print('This is an example of a positive tweet: \n', train_x[0])
print('\nThis is an example of the processed version of the tweet: \n', process_text(train_x[0]))

This is an example of a positive tweet: 
 Easy to use, economical!

This is an example of the processed version of the tweet: 
 ['easi', 'use', 'econom']


<a name='1'></a>
## 1 - Logistic Regression 

<a name='1-1'></a>
### 1.1 - Sigmoid
You will learn to use logistic regression for text classification. 
* The sigmoid function is defined as: 

$$ h(z) = \frac{1}{1+\exp^{-z}} \tag{1}$$

It maps the input 'z' to a value that ranges between 0 and 1, and so it can be treated as a probability. 

<div style="width:image width px; font-size:100%; text-align:center;"><img src='./images/sigmoid_plot.jpg' alt="alternate text" width="width" height="height" style="width:300px;height:200px;" /> Figure 1 </div>

In [18]:
def sigmoid(z): 
    '''
    Inputs:
        z: is the input (can be a scalar or an array)
    Outputs:
        h: the sigmoid of z
    '''
    # calculate the sigmoid of z
    h = 1/(1+np.exp(-z))
    
    return h

#### Logistic Regression: Regression and a Sigmoid

Logistic regression takes a regular linear regression, and applies a sigmoid to the output of the linear regression.

Regression:
$$z = \theta_0 x_0 + \theta_1 x_1 + \theta_2 x_2 + ... \theta_N x_N$$
Note that the $\theta$ values are "weights". If you took the deep learning specialization, we referred to the weights with the 'w' vector.  In this course, we're using a different variable $\theta$ to refer to the weights.

Logistic regression
$$ h(z) = \frac{1}{1+\exp^{-z}}$$
$$z = \theta_0 x_0 + \theta_1 x_1 + \theta_2 x_2 + ... \theta_N x_N$$
We will refer to 'z' as the 'logits'.

<a name='1-2'></a>
### 1.2 - Cost function and Gradient

The cost function used for logistic regression is the average of the log loss across all training examples:

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^m y^{(i)}\log (h(z(\theta)^{(i)})) + (1-y^{(i)})\log (1-h(z(\theta)^{(i)}))\tag{5} $$
* $m$ is the number of training examples
* $y^{(i)}$ is the actual label of training example 'i'.
* $h(z^{(i)})$ is the model's prediction for the training example 'i'.

The loss function for a single training example is
$$ Loss = -1 \times \left( y^{(i)}\log (h(z(\theta)^{(i)})) + (1-y^{(i)})\log (1-h(z(\theta)^{(i)})) \right)$$

* All the $h$ values are between 0 and 1, so the logs will be negative. That is the reason for the factor of -1 applied to the sum of the two loss terms.
* Note that when the model predicts 1 ($h(z(\theta)) = 1$) and the label 'y' is also 1, the loss for that training example is 0. 
* Similarly, when the model predicts 0 ($h(z(\theta)) = 0$) and the actual label is also 0, the loss for that training example is 0. 
* However, when the model prediction is close to 1 ($h(z(\theta)) = 0.9999$) and the label is 0, the second term of the log loss becomes a large negative number, which is then multiplied by the overall factor of -1 to convert it to a positive loss value. $-1 \times (1 - 0) \times log(1 - 0.9999) \approx 9.2$ The closer the model prediction gets to 1, the larger the loss.

In [19]:
# verify that when the model predicts close to 1, but the actual label is 0, the loss is a large positive value
-1 * (1 - 0) * np.log(1 - 0.9999) # loss is about 9.2

np.float64(9.210340371976294)

* Likewise, if the model predicts close to 0 ($h(z) = 0.0001$) but the actual label is 1, the first term in the loss function becomes a large number: $-1 \times log(0.0001) \approx 9.2$.  The closer the prediction is to zero, the larger the loss.

In [20]:
# verify that when the model predicts close to 0 but the actual label is 1, the loss is a large positive value
-1 * np.log(0.0001) # loss is about 9.2

np.float64(9.210340371976182)

#### Update the weights

To update your weight vector $\theta$, you will apply gradient descent to iteratively improve your model's predictions.  
The gradient of the cost function $J$ with respect to one of the weights $\theta_j$ is:

$$\nabla_{\theta_j}J(\theta) = \frac{1}{m} \sum_{i=1}^m(h^{(i)}-y^{(i)})x^{(i)}_j \tag{5}$$
* 'i' is the index across all 'm' training examples.
* 'j' is the index of the weight $\theta_j$, so $x^{(i)}_j$ is the feature associated with weight $\theta_j$

* To update the weight $\theta_j$, we adjust it by subtracting a fraction of the gradient determined by $\alpha$:
$$\theta_j = \theta_j - \alpha \times \nabla_{\theta_j}J(\theta) $$
* The learning rate $\alpha$ is a value that we choose to control how big a single update will be.


<a name='ex-2'></a>
### Exercise 2 - gradientDescent
Implement gradient descent function.
* The number of iterations 'num_iters" is the number of times that you'll use the entire training set.
* For each iteration, you'll calculate the cost function using all training examples (there are 'm' training examples), and for all features.
* Instead of updating a single weight $\theta_i$ at a time, we can update all the weights in the column vector:  
$$\mathbf{\theta} = \begin{pmatrix}
\theta_0
\\
\theta_1
\\ 
\theta_2 
\\ 
\vdots
\\ 
\theta_n
\end{pmatrix}$$
* $\mathbf{\theta}$ has dimensions (n+1, 1), where 'n' is the number of features, and there is one more element for the bias term $\theta_0$ (note that the corresponding feature value $\mathbf{x_0}$ is 1).
* The 'logits', 'z', are calculated by multiplying the feature matrix 'x' with the weight vector 'theta'.  $z = \mathbf{x}\mathbf{\theta}$
    * $\mathbf{x}$ has dimensions (m, n+1) 
    * $\mathbf{\theta}$: has dimensions (n+1, 1)
    * $\mathbf{z}$: has dimensions (m, 1)
* The prediction 'h', is calculated by applying the sigmoid to each element in 'z': $h(z) = sigmoid(z)$, and has dimensions (m,1).
* The cost function $J$ is calculated by taking the dot product of the vectors 'y' and 'log(h)'.  Since both 'y' and 'h' are column vectors (m,1), transpose the vector to the left, so that matrix multiplication of a row vector with column vector performs the dot product.
$$J = \frac{-1}{m} \times \left(\mathbf{y}^T \cdot log(\mathbf{h}) + \mathbf{(1-y)}^T \cdot log(\mathbf{1-h}) \right)$$
* The update of theta is also vectorized.  Because the dimensions of $\mathbf{x}$ are (m, n+1), and both $\mathbf{h}$ and $\mathbf{y}$ are (m, 1), we need to transpose the $\mathbf{x}$ and place it on the left in order to perform matrix multiplication, which then yields the (n+1, 1) answer we need:
$$\mathbf{\theta} = \mathbf{\theta} - \frac{\alpha}{m} \times \left( \mathbf{x}^T \cdot \left( \mathbf{h-y} \right) \right)$$

In [21]:
def gradientDescent(x, y, theta, alpha, num_iters):
    '''
    Inputs:
        x: matrix of features which is (m,n+1)
        y: corresponding labels of the input matrix x, dimensions (m,1)
        theta: weight vector of dimension (n+1,1)
        alpha: learning rate
        num_iters: number of iterations you want to train your model for
    Outputs:
        J: the final cost
        theta: your final weight vector
    '''
    # get 'm', the number of rows in matrix x
    m = len(x)  
    for i in range(0, num_iters):
        
        # get z, the dot product of x and theta
        z = np.dot(x,theta)
        
        # get the sigmoid of z
        h = sigmoid(z)
        
        # calculate the cost function
        J = -1/m*(np.dot(y.transpose(),np.log(h))+np.dot((1-y).transpose(),np.log(1-h)))

        # update the weights theta
        theta = theta -alpha/m*(np.dot(x.transpose(),(h-y)))
    J = float(J)
    return J, theta

In [22]:
# Check the function
# Construct a synthetic test case using numpy PRNG functions
np.random.seed(1)
# X input is 10 x 3 with ones for the bias terms
tmp_X = np.append(np.ones((10, 1)), np.random.rand(10, 2) * 2000, axis=1)
# Y Labels are 10 x 1
tmp_Y = (np.random.rand(10, 1) > 0.35).astype(float)

# Apply gradient descent
tmp_J, tmp_theta = gradientDescent(tmp_X, tmp_Y, np.zeros((3, 1)), 1e-8, 700)
print(f"The cost after training is {tmp_J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(tmp_theta)]}")

The cost after training is 0.67094970.
The resulting vector of weights is [np.float64(4.1e-07), np.float64(0.00035658), np.float64(7.309e-05)]


/var/folders/vp/0c9j9j5j5ns2r2w6p14z6t2h0000gn/T/ipykernel_32790/3197981203.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


In [23]:
# Check the gradient descent function
np.random.seed(1)
# X input is 20 x 3 with ones for the bias terms
tmp_X = np.append(np.ones((20, 1)), np.random.rand(20, 2) * 2000, axis=1)
# Y Labels are 20 x 1
tmp_Y = (np.random.rand(20, 1) > 0.35).astype(float)

# Apply gradient descent
tmp_J, tmp_theta = gradientDescent(tmp_X, tmp_Y, np.zeros((3, 1)), 1e-8, 1000) 
# you can check gradient descent performance with different number of iterations (300, 500, 700, 1000, 1200, 1500)
print(f"The cost after training is {tmp_J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(tmp_theta)]}")

The cost after training is 0.66111809.
The resulting vector of weights is [np.float64(-3.9e-07), np.float64(-0.0003557), np.float64(0.00039862)]


/var/folders/vp/0c9j9j5j5ns2r2w6p14z6t2h0000gn/T/ipykernel_32790/3197981203.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


<a name='2'></a>
## 2 - Extracting the Features

* Given a list of reviews, extract the features and store them in a matrix.
    * The first feature is the number of positive words in a review.
    * The second feature is the number of negative words in a review. 
* Then train your logistic regression classifier on these features.
* Test the classifier on a validation set. 


In [24]:
def extract_features(tweet, freqs, process_textt=process_text):
    '''
    Input: 
        tweet: a string containing one tweet
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
    Output: 
        x: a feature vector of dimension (1,3)
    '''
    # process_tweet tokenizes, stems, and removes stopwords
    word_l = process_text(tweet)
    
    # 3 elements for [bias, positive, negative] counts
    x = np.zeros(3) 
    
    # bias term is set to 1
    x[0] = 1 
    
    # loop through each word in the list of words
    for word in word_l:
        
        # increment the word count for the positive label 1
        x[1] += freqs.get((word, 1),0)
        
        # increment the word count for the negative label 0
        x[2] += freqs.get((word, 0),0)
    
    x = x[None, :]  # adding batch dimension for further processing
    assert(x.shape == (1, 3))
    return x

In [25]:
# Check your function
tmp1 = extract_features(train_x[0], freqs)
print(tmp1)

[[1.00e+00 7.73e+03 9.06e+02]]


In [26]:
# test 2:
# check for when the words are not in the freqs dictionary
tmp2 = extract_features('bfhuehv blfeej34b bloodecweb', freqs)
print(tmp2)

[[1. 0. 0.]]


<a name='3'></a>
## 3 - Training Your Model

To train the model:
* Stack the features for all training examples into a matrix X. 
* Call `gradientDescent`, which you've implemented above.


In [27]:
# collect the features 'x' and stack them into a matrix 'X'
X = np.zeros((len(train_x), 3))
for i in range(len(train_x)):
    X[i, :]= extract_features(train_x[i], freqs)

# training labels corresponding to X
Y = train_y

# Apply gradient descent
J, theta = gradientDescent(X, Y, np.zeros((3, 1)), 1e-9, 6500)
print(f"The cost after training is {J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(theta)]}")

The cost after training is 0.42432647.
The resulting vector of weights is [np.float64(-5.4e-07), np.float64(0.0004767), np.float64(-0.00089652)]


/var/folders/vp/0c9j9j5j5ns2r2w6p14z6t2h0000gn/T/ipykernel_32790/3197981203.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


<a name='4'></a>
## 4 -  Test your Logistic Regression

Finally we have to test logistic regression. 
<a name='ex-4'></a>

Implement `predict_review`.
Predict whether a review is positive or negative.

* Given a review, process it, then extract the features.
* Apply the model's learned weights on the features to get the logits.
* Apply the sigmoid to the logits to get the prediction (a value between 0 and 1).

$$y_{pred} = sigmoid(\mathbf{x} \cdot \theta)$$

In [28]:
def predict_review(tweet, freqs, theta):
    '''
    Input: 
        tweet: a string
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
        theta: (3,1) vector of weights
    Output: 
        y_pred: the probability of a tweet being positive or negative
    '''    
    # extract the features of the tweet and store it into x
    x = extract_features(tweet, freqs)
    
    # make the prediction using x and theta
    y_pred = sigmoid(np.dot(x,theta))
    
    return y_pred

In [29]:
# Run this cell to test your function
for tweet in ['The easiest device I have used until now', 'very bad photo device, i can not use it']:
    print( '%s -> %f' % (tweet, predict_review(tweet, freqs, theta)))    
    

The easiest device I have used until now -> 0.753801
very bad photo device, i can not use it -> 0.731418


/var/folders/vp/0c9j9j5j5ns2r2w6p14z6t2h0000gn/T/ipykernel_32790/2636916858.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print( '%s -> %f' % (tweet, predict_review(tweet, freqs, theta)))


In [30]:
# Feel free to check the sentiment of your own tweet below
my_review = 'I am learning to use my latest state of the art camera:)'
my_review_neg = 'Horrible device. not working'
predict_review(my_review, freqs, theta)
predict_review(my_review_neg, freqs, theta)

array([[0.44940873]])

<a name='4-1'></a>
### 4.1 -  Check the Performance using the Test Set
After training your model using the training set above, we have to check the model accuracy

<a name='ex-5'></a>

Implement `test_logistic_regression`. 
* Given the test data and the weights of your trained model, calculate the accuracy of your logistic regression model. 
* Use your 'predict_tweet' function to make predictions on each tweet in the test set.
* If the prediction is > 0.5, set the model's classification 'y_hat' to 1, otherwise set the model's classification 'y_hat' to 0.
* A prediction is accurate when the y_hat equals the test_y.  Sum up all the instances when they are equal and divide by m.


In [31]:
def test_logistic_regression(test_x, test_y, freqs, theta, predict_review=predict_review):
    """
    Input: 
        test_x: a list of reviews
        test_y: (m, 1) vector with the corresponding labels for the list of reviews
        freqs: a dictionary with the frequency of each pair (or tuple)
        theta: weight vector of dimension (3, 1)
    Output: 
        accuracy: (# of tweets classified correctly) / (total # of reviews)
    """
    
    # the list for storing predictions
    y_hat = []
    
    for review in test_x:
        # get the label prediction for the tweet
        y_pred = predict_review(review, freqs, theta)
        
        if y_pred > 0.5:
            # append 1.0 to the list
            y_hat.append(1.0)
        else:
            # append 0 to the list
            y_hat.append(0.0)

    # With the above implementation, y_hat is a list, but test_y is (m,1) array
    # convert both to one-dimensional arrays in order to compare them using the '==' operator
    y_hat = np.array(y_hat)[:,np.newaxis]
    accuracy = np.sum(y_hat == test_y)/len(test_y)
    
    return accuracy

In [32]:
tmp_accuracy = test_logistic_regression(test_x, test_y, freqs, theta)
print(f"Logistic regression model's accuracy = {tmp_accuracy:.4f}")

Logistic regression model's accuracy = 0.7629


Later in this specialization, we will see how we can use deeplearning to improve the prediction performance.

<a name='5'></a>
## 5 - Predict with your own text

In [33]:
# Feel free to change the tweet below
my_review = 'The resolution of my camera is bad! I do not like it ... bad bad bad!'
print(process_text(my_review))
y_hat = predict_review(my_review, freqs, theta)
print(y_hat)
if y_hat > 0.5:
    print('Positive sentiment')
else: 
    print('Negative sentiment')

['resolut', 'camera', 'bad', 'like', '...', 'bad', 'bad', 'bad']
[[0.17683306]]
Negative sentiment
